In [17]:
import carla
import time
import pygame
import logging

In [19]:
CARLA_LOCALHOST = "localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT = 20.0

IMAGE_Z = 3
IMAGE_X = 0

camera_init_trans = carla.Transform(carla.Location(z = IMAGE_Z, x = IMAGE_X))

In [21]:
def connect_to_carla() -> tuple[carla.Client, carla.World]:

    client = carla.Client(CARLA_LOCALHOST, CARLA_PORT)
    client.set_timeout(20)
    world = client.get_world()

    return client, world

In [23]:
def setup(world: carla.World) -> None:

    logging.basicConfig(
        level = logging.INFO,
        format ="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
    )

    logger = logging.getLogger(__name__)
    
    try:
        for actor in world.get_actor().filter("vehicle.*"):
            actor.destroy()
    except:
        logger.info("No vehicle found, during setup.")
    finally:
        logger.info("Scene is clear.")



In [22]:
def spawn_vehicle(world: carla.World) -> carla.Vehicle:

    actors = world.get_actors().filter("vehicle.*")

    for vehicle in actors:
        vehicle.destroy()

    vehicle_bp = world.get_blueprint_library().filter("*cybertruck*")

    if not vehicle_bp:
        print("No Cybertrunk blueprint was found !")
        return None
    
    spawn_points = world.get_map().get_spawn_points()

    vehicle = world.try_spawn_actor(
        vehicle_bp[0], 
        spawn_points[0]
    )

    if vehicle is None:
        print("Failed to spawn vehicle !")
        return None

    return vehicle

In [31]:
def spawn_rgb_camera(world: carla.World, vehicle: carla.Vehicle) -> carla.Sensor:

    rbg_bp = world.get_blueprint_library().find("sensor.camera.rgb")

    if not rbg_bp:
        raise RuntimeError(
            "No valid RGB blueprint found."
        )

    rgb=world.spawn_actor(rbg_bp, camera_init_trans, attach_to=vehicle)
    
    return rgb

In [ ]:
def control_vehicle(vehicle):

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
    )

    logger = logging.getLogger(__name__)

    pygame.init()

    screen = pygame.display.set_mode((400, 200))
    pygame.display.set_caption("CARLA Manual Control")

    clock = pygame.time.Clock()

    running = True

    while running:

        # Check keyboard events
        for event in pygame.event.get():

            if event.type == pygame.QUIT:
                running = False

            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_q:
                    running = False

        # Get current keyboard state
        keys = pygame.key.get_pressed()

        # Create a fresh control object every iteration
        control = carla.VehicleControl()

        # Throttle and brake
        if keys[pygame.K_w]:
            control.throttle = 0.4

        if keys[pygame.K_s]:
            control.brake = 0.1

        # Steering
        if keys[pygame.K_a]:
            control.steer = -0.5
        elif keys[pygame.K_d]:
            control.steer = 0.5

        # Apply control continuously
        vehicle.apply_control(control)

        clock.tick(60)

    # Stop vehicle when exiting
    vehicle.apply_control(carla.VehicleControl())

    pygame.quit()


In [6]:
def set_weather_conditions(world):
    world.set_weather(carla.WeatherParameters.ClearNight)

In [26]:
def end_world(world, vehicle):

    vehicles = world.get_actors().filter("vehicle.*")
    for vehicle in vehicles:
        vehicle.destroy()

    print(f"Destroyed {len(vehicles)} vehicles.")

In [ ]:
def rgb_callback(data, data_dict):
    data_dict["rgb_data"] = 

In [27]:
def main():
    client, world = connect_to_carla()
    setup(world)
    #set_weather_conditions(world)
    vehicle = spawn_vehicle(world)
    rgb = spawn_rgb_camera(world, vehicle)
    control_vehicle(vehicle)
    
    
   
    

In [32]:
if __name__ == "__main__":
    main()
    

2026-11-12 23:06:13,447 [INFO] __main__: No vehicle found, during setup.
2026-11-12 23:06:13,449 [INFO] __main__: Scene is clear.
